# Phase 6: Predictive Failure Modeling

This notebook completes Phase 6 of the Bosch roadmap:

30. Create train-validation-test datasets.
31. Build Logistic Regression baseline.
32. Build Random Forest model.
33. Build XGBoost model.
34. Build LightGBM model.
35. Build CatBoost model.
36. Compare MCC, Precision, Recall, F1, and PR-AUC.
37. Select best-performing model.

The pipeline uses raw numeric, categorical, and date-derived train/test data. Before modeling, it profiles feature correlation and missingness so important missing-value signals are kept as model features instead of being blindly removed.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

PROJECT_ROOT

## Run The Phase 6 Pipeline

This script is the reusable source of truth for Phase 6. It profiles raw numeric and categorical data, builds model-ready train/validation datasets, scores the Kaggle test rows, trains all requested algorithms, and writes reports/model artifacts. Existing correlation/model artifacts are reused on rerun where possible.

In [ ]:
import subprocess

script_path = PROJECT_ROOT / 'src' / 'data' / 'phase6_predictive_failure_modeling.py'
subprocess.run([sys.executable, str(script_path)], cwd=PROJECT_ROOT, check=True)

## Load Outputs

The output files below are intentionally small enough to inspect in the notebook, while the full train/validation model tables and test predictions are saved in `data/processed`.

In [ ]:
import pandas as pd

processed_dir = PROJECT_ROOT / 'data' / 'processed'
reports_dir = PROJECT_ROOT / 'reports'

numeric_corr = pd.read_csv(reports_dir / 'phase6_numeric_correlation_report.csv')
categorical_presence = pd.read_csv(reports_dir / 'phase6_categorical_presence_report.csv')
final_corr = pd.read_csv(reports_dir / 'phase6_final_feature_correlation_report.csv')
model_metrics = pd.read_csv(reports_dir / 'phase6_model_comparison_metrics.csv')
test_predictions = pd.read_csv(processed_dir / 'phase6_test_predictions.csv')

model_metrics

## Train, Validation, And Test Dataset Checks

The training dataset contains all failure rows and a sampled set of non-failure rows, which keeps the rare-failure problem practical while preserving the full positive class. The Kaggle test rows are scored separately because they do not include `Response`.

In [ ]:
train_cols = pd.read_csv(processed_dir / 'phase6_train_dataset.csv', nrows=0).columns
valid_cols = pd.read_csv(processed_dir / 'phase6_validation_dataset.csv', nrows=0).columns
test_preview = pd.read_csv(processed_dir / 'phase6_test_dataset_preview.csv')

print(f'Train feature columns including Id/Response: {len(train_cols):,}')
print(f'Validation feature columns including Id/Response: {len(valid_cols):,}')
print(f'Test predictions: {len(test_predictions):,}')
test_preview.head()

## Correlation Before Modeling

The next tables show which features were most related to the target before training. Missingness is kept when it carries signal because a missing Bosch station value often means the part did not pass through that station.

In [ ]:
numeric_corr.head(15)

In [ ]:
categorical_presence.head(15)

In [ ]:
final_corr.head(20)

## Model Comparison

The selected threshold for each model is the validation threshold that maximizes MCC. MCC is useful here because the target is highly imbalanced.

In [ ]:
model_metrics.sort_values(['rank'])

## Selected Model And Test Predictions

The best model is saved as `models/phase6_best_model.joblib`. The full test prediction file contains one row per Kaggle test product with a failure probability and a thresholded failure flag.

In [ ]:
best_model = model_metrics.sort_values('rank').iloc[0]
print(best_model.to_string())
test_predictions.head(10)

## Phase 6 Summary

This phase creates reproducible modeling artifacts using all three Bosch data types. LightGBM is selected as the best model in this run, ranked by MCC first and PR-AUC second.